In [6]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

In [7]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location=('/workspaces/MLOps-ZoomCamp/02-Experiment tracking and model '
 'management/mlruns/1'), creation_time=1748500545859, experiment_id='1', last_update_time=1748500545859, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [8]:
def read_dataframe(filename):
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)

        df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
        df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    elif filename.endswith(".parquet"):
        df = pd.read_parquet(filename)

    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

In [9]:
df_train = read_dataframe("../data/yellow_tripdata_2023-01.parquet")
df_val = read_dataframe("../data/yellow_tripdata_2023-02.parquet")

In [10]:
categorical = ["PULocationID", "DOLocationID"]

# Turn dataframes into list of dictionaries
train_dicts = df_train[categorical].to_dict(orient="records")
val_dicts = df_val[categorical].to_dict(orient="records")

In [11]:
# Set GT Values
y_train = df_train["duration"].values
y_val = df_val["duration"].values

In [12]:
# Fit dictionary vectorizer on Training data
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [15]:
# Train linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [16]:
# Apply learned dictionary vectorizer on validation data
X_val = dv.transform(val_dicts)

# Make predictions on validation data
y_pred = lr.predict(X_val)

# Calculate RMSE on validation
rmse = root_mean_squared_error(y_val, y_pred)

In [17]:
rmse

7.811817745843695

In [16]:
with mlflow.start_run():
    mlflow.set_tag("developer", "Wahba")

    mlflow.log_param("train-data-path", "../data/yellow_tripdata_2023-01.parquet")
    mlflow.log_param("valid-data-path", "../data/yellow_tripdata_2023-02.parquet")

    alpha = 0.001
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

In [18]:
# trying xgboost
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [19]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [19]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50,
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {"loss": rmse, "status": STATUS_OK}

In [ ]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:linear",
    "seed": 42,
}

best_result = fmin(
    fn=objective, space=search_space, algo=tpe.suggest, max_evals=50, trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/home/codespace/anaconda3/envs/mlflow_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [07:35:50] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.39455                           
[1]	validation-rmse:8.92939                           
[2]	validation-rmse:8.60413                           
[3]	validation-rmse:8.37730                           
[4]	validation-rmse:8.21755                           
[5]	validation-rmse:8.09758                           
[6]	validation-rmse:8.01392                           
[7]	validation-rmse:7.93536                           
[8]	validation-rmse:7.87804                           
[9]	validation-rmse:7.83616                           
[10]	validation-rmse:7.79834                          
[11]	validation-rmse:7.75820                          
[12]	validation-rmse:7.72292                          
[13]	validation-rmse:7.69903                          
[14]	validation-rmse:7.61648                          
[15]	validation-rmse:7.59689                          
[16]	validation-rmse:7.52547                          
[17]	validation-rmse:7.51170                          
[18]	valid

/home/codespace/anaconda3/envs/mlflow_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [07:42:25] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.19991                                                       
[1]	validation-rmse:8.58449                                                       
[2]	validation-rmse:7.96300                                                       
[3]	validation-rmse:7.65937                                                       
[4]	validation-rmse:7.42856                                                       
[5]	validation-rmse:7.28202                                                       
[6]	validation-rmse:6.96035                                                       
[7]	validation-rmse:6.88137                                                       
[8]	validation-rmse:6.81958                                                       
[9]	validation-rmse:6.58691                                                       
[10]	validation-rmse:6.54710                                                      
[11]	validation-rmse:6.51486                                                      
[12]

...

[0]	validation-rmse:8.30988                                                         
[1]	validation-rmse:6.77357                                                         
[2]	validation-rmse:6.13321                                                         
[3]	validation-rmse:5.93337                                                         
[4]	validation-rmse:5.75278                                                         
[5]	validation-rmse:5.68716                                                         
[6]	validation-rmse:5.64996                                                         
[7]	validation-rmse:5.60712                                                         
[8]	validation-rmse:5.58098                                                         
[9]	validation-rmse:5.40815                                                         
[10]	validation-rmse:5.40013                                                        
[11]	validation-rmse:5.37464                                     

/home/codespace/anaconda3/envs/mlflow_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [09:35:40] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.58751                                                           
[1]	validation-rmse:9.18178                                                           
[2]	validation-rmse:8.83378                                                           
[3]	validation-rmse:8.48294                                                           
[4]	validation-rmse:8.23112                                                           
[5]	validation-rmse:7.86537                                                           
[6]	validation-rmse:7.56170                                                           
[7]	validation-rmse:7.40578                                                           
[8]	validation-rmse:7.17280                                                           
[9]	validation-rmse:7.06519                                                           
[10]	validation-rmse:6.86780                                                          
[11]	validation-rmse:6.79313               

/home/codespace/anaconda3/envs/mlflow_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [09:50:58] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:8.67602                                                            
[1]	validation-rmse:7.51838                                                            
[2]	validation-rmse:7.00024                                                            
[3]	validation-rmse:6.29643                                                            
[4]	validation-rmse:6.10317                                                            
[5]	validation-rmse:5.76444                                                            
[6]	validation-rmse:5.69724                                                            
[7]	validation-rmse:5.52829                                                            
[8]	validation-rmse:5.50026                                                            
[9]	validation-rmse:5.47506                                                            
[10]	validation-rmse:5.39691                                                           
[11]	validation-rmse:5.38617    

In [20]:
import mlflow.xgboost


params = {
    "learning_rate": 0.37423890981602775,
    "max_depth": 80,
    "min_child_weight": 2.4679201221277838,
    "objective": "reg:linear",
    "reg_alpha": 0.008474358456426086,
    "reg_lambda": 0.006361637376418637,
    "seed": 42,
}

# autolog for xgboost
mlflow.xgboost.autolog()

booster = xgb.train(
    params=params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(valid, "validation")],
    early_stopping_rounds=50,
)

2025/05/29 16:45:28 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9d7cf58bcc08493e9ec7068af5b73c6b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


: 